# Sigiriya SinLlama evidence handoff v4

Keep the existing evidence JSON for a fair comparison. From the Windows project directory, regenerate only the runtime ZIP.

```powershell
.\.venv\Scripts\python.exe -B -X utf8 -m scripts.sinllama_handoff --runtime-only
```

Use **Runtime > Change runtime type > T4 GPU**. Run the notebook in a fresh runtime. Upload `sigiriya_sinllama_evidence.json` and `sigiriya_sinllama_runtime_v4.zip` from `results/generation/`. Neo4j and E5 stay on Windows.

This [Sinhala base completion model](https://huggingface.co/SAWithanage/SinLlama-Llama-3-8B-Merged) uses plain completion prompts. The verifier still checks complete, ordered copies of the selected excerpts. The optional experimental verifier checks each claim and can recover a complete supported prefix.


## 1. Check the GPU and install inference dependencies

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")
print(torch.cuda.get_device_name(0), torch.__version__)


In [ ]:
%pip install -q transformers==4.57.1 accelerate==1.10.1 bitsandbytes==0.48.2


## 2. Upload the exported evidence and runtime ZIP

In [ ]:
import json
import sys
from pathlib import Path
from zipfile import ZipFile

from google.colab import files

uploaded = files.upload()
evidence_name = "sigiriya_sinllama_evidence.json"
runtime_name = "sigiriya_sinllama_runtime_v4.zip"
if not {evidence_name, runtime_name} <= uploaded.keys():
    raise ValueError("Upload both files produced by the local exporter.")

runtime = Path("/content/sigiriya_sinllama_runtime")
expected = {
    "src/__init__.py", "src/config.py", "src/generation.py", "src/prompting.py",
    "scripts/sinllama_handoff.py", "src/verification.py",
}
with ZipFile(runtime_name) as archive:
    if set(archive.namelist()) != expected:
        raise ValueError("Unexpected runtime files. Re-export from the local project.")
    archive.extractall(runtime, members=sorted(expected))
sys.path.insert(0, str(runtime))
payload = json.loads(Path(evidence_name).read_text(encoding="utf-8-sig"))
print(f"Loaded {len(payload['questions'])} questions.")


## 3. Generate and display each question

Leave **QUESTION_NUMBER** at **0** for all exported questions. Enter **1–9** for one of the standard questions. **PROFILE** defaults to **sinhala_system_grounded** in this notebook. Choose **strict** or **experimental** verification independently. Each question and its result print immediately. The generated preview is shortened for display and the complete raw completion remains in JSON.


In [ ]:
from importlib.metadata import version

from scripts.sinllama_handoff import (
    COMPARISON_PROFILES, format_question_result, generate_batch, summarize_generation, write_json,
)
from src.generation import SinLlamaGenerator

QUESTION_NUMBER = 0  # @param {type:"integer"}
PROFILE = "sinhala_system_grounded"  # @param ["source_qa", "sinhala_qa", "sinhala_system_grounded", "short_grounding", "source_qa_penalty_1_15", "few_shot"]
VERIFICATION_MODE = "experimental"  # @param ["strict", "experimental"]

if "generator" not in globals():
    generator = SinLlamaGenerator(local_files_only=False)
generator.config = COMPARISON_PROFILES[PROFILE]
runtime_info = {
    "gpu": torch.cuda.get_device_name(0),
    "versions": {pkg: version(pkg) for pkg in ("torch", "transformers", "accelerate", "bitsandbytes")},
}
output = generate_batch(
    payload, generator, question_number=QUESTION_NUMBER or None,
    verification_mode=VERIFICATION_MODE,
    on_result=lambda number, row: print(format_question_result(number, row), flush=True),
)
output["runtime"] = runtime_info
output["profile"] = PROFILE
output["summary"] = summarize_generation(output["results"])
suffix = f"_q{QUESTION_NUMBER:02d}" if QUESTION_NUMBER else ""
result_path = Path(f"sigiriya_sinllama_results_v4{suffix}.json")
write_json(result_path, output)
print(output["summary"])


## 4. Download the v4 result

In [ ]:
files.download(str(result_path))


## 5. Compare the Sinhala prompts on Q01, Q03 and Q05

Set **RUN_COMPARISON** to true. The default comparison uses **sinhala_qa** and **sinhala_system_grounded** with identical decoding settings and the same selected excerpts. The comparison records strict and experimental decisions for each completion. Enable **ALL_PROFILES** to include the earlier prompt and penalty experiments. Optionally upload a historical result JSON for these same questions and excerpts.

Review Q01 for accurate interpretation of the figures and preservation of uncertainty. Review Q03 for coverage of both the frescoes and the Mirror Wall. Review Q05 for underground channels, gravity, and water pressure, including **බව සැලකේ**, and check for invented details. Inspect raw completions and discarded continuations before assessing quality.


In [ ]:
from scripts.sinllama_handoff import FOCUS_QUESTIONS, SYSTEM_PROMPT_PROFILES, compare_generation

RUN_COMPARISON = False  # @param {type:"boolean"}
ALL_PROFILES = False  # @param {type:"boolean"}
UPLOAD_BASELINE = False  # @param {type:"boolean"}

if RUN_COMPARISON:
    baseline = None
    if UPLOAD_BASELINE:
        uploaded_baseline = files.upload()
        if len(uploaded_baseline) != 1:
            raise ValueError("Upload one historical results JSON.")
        baseline_bytes, = uploaded_baseline.values()
        baseline = json.loads(baseline_bytes.decode("utf-8-sig"))

    def show_comparison(profile, number, row):
        print(f"=== {profile} ===")
        print(format_question_result(number, row), flush=True)

    comparison = compare_generation(
        payload, generator, baseline=baseline, on_result=show_comparison,
        question_numbers=FOCUS_QUESTIONS,
        profile_names=None if ALL_PROFILES else SYSTEM_PROMPT_PROFILES,
    )
    comparison["runtime"] = runtime_info
    comparison_path = Path("sinllama_prompt_comparison_v4.json")
    write_json(comparison_path, comparison)
    for profile, run in comparison["runs"].items():
        print(profile, run["summary"])
    files.download(str(comparison_path))
